# Lakehouse Veri Keşfi ve İlişkiler

Bu notebook, `analytics.duckdb` içindeki verileri read-only olarak inceler.
Amaç: hangi veri ailelerinin bulunduğunu, tabloların kolonlarını, mantıksal foreign key ilişkilerini, eksiklikleri ve basit analiz örneklerini somutlaştırmak.

Not: DuckDB içinde fiziksel FOREIGN KEY constraint'leri tanımlı olmayabilir. Aşağıdaki ilişkiler kolon isimleri ve veri sözlükleri üzerinden mantıksal olarak test edilir.

Notebook lakehouse dosyasının mevcut snapshot’ını kullanır. Lakehouse yeniden build edilirse notebook aynı path’ten yeni DuckDB içeriğini okuyacaktır.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from IPython.display import display

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'data_pipeline' / 'lakehouse' / 'analytics.duckdb').exists():
        repo_root = candidate
        break
DB_PATH = repo_root / 'data_pipeline' / 'lakehouse' / 'analytics.duckdb'
con = duckdb.connect(str(DB_PATH), read_only=True)
print(f'DuckDB: {DB_PATH}')
print('Bağlantı: read-only')

DuckDB: /home/neo/Desktop/GITHUB MYZ21/agentic-data-analytics/data_pipeline/lakehouse/analytics.duckdb
Bağlantı: read-only


## 1. Lakehouse genel görünümü
Schema, tablo ve satır sayılarını inceleyelim.

In [2]:
schemas = con.sql("""
    SELECT DISTINCT table_schema AS schema_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY schema_name
""").df()
tables = con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").df()
tables['qualified_name'] = tables['table_schema'] + '.' + tables['table_name']
tables['row_count'] = tables['qualified_name'].map(lambda name: con.sql(f'SELECT count(*) AS n FROM {name}').fetchone()[0])
print(f'Schema sayısı: {len(schemas)} | Tablo sayısı: {len(tables)}')
display(schemas)
display(tables[['qualified_name', 'row_count']])

Schema sayısı: 9 | Tablo sayısı: 60


,schema_name
0,analysis
1,bddk
2,catalog
3,evds
4,evidence
5,quality
6,regional
7,tbb
8,tuik


,qualified_name,row_count
0,analysis.housing_credit_monthly,66
1,analysis.housing_credit_quarterly,22
2,bddk.finturk_branch_zero_fallback_audit,1328
3,bddk.finturk_measurements,936512
4,bddk.finturk_metric_dictionary,76
5,bddk.finturk_missingness_audit,47
6,bddk.finturk_source_tables,154
7,bddk.housing_credit_monthly,66
8,bddk.monthly_measurements,1334850
9,bddk.monthly_metric_dictionary,2108


In [3]:
storage = con.sql('PRAGMA database_size').df()
display(storage)
display(tables.groupby('table_schema', as_index=False)['row_count'].sum().sort_values('row_count', ascending=False))

,database_name,database_size,block_size,total_blocks,used_blocks,free_blocks,wal_size,memory_usage,memory_limit
0,analytics,98.5 MiB,262144,394,394,0,0 bytes,6.5 MiB,18.5 GiB


,table_schema,row_count
1,bddk,3319323
3,evds,143152
8,tuik,56872
2,catalog,55619
6,regional,1894
7,tbb,953
0,analysis,88
5,quality,22
4,evidence,5


## 2. Kolonlar, tipler ve NULL oranları
Her tablo için kolon adlarını ve veri tiplerini gösteriyoruz. Büyük tablolar için NULL oranı ayrıca raporlanıyor.

In [4]:
columns = con.sql("""
    SELECT table_schema, table_name, ordinal_position, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name, ordinal_position
""").df()
for qualified_name, group in columns.groupby(['table_schema', 'table_name']):
    schema_name, table_name = qualified_name
    print(f'\n### {schema_name}.{table_name}')
    display(group[['ordinal_position', 'column_name', 'data_type']])


### analysis.housing_credit_monthly


,ordinal_position,column_name,data_type
0,1,month,VARCHAR
1,2,bank_group_name_tr,VARCHAR
2,3,metric_name_tr,VARCHAR
3,4,tp_million_tl,BIGINT
4,5,yp_million_tl,BIGINT
...,...,...,...
85,86,rate_down_real_stock_not_up_quality_screened,BOOLEAN
86,87,event_count,BIGINT
87,88,event_ids,VARCHAR
88,89,event_titles,VARCHAR



### analysis.housing_credit_quarterly


,ordinal_position,column_name,data_type
90,1,quarter,VARCHAR
91,2,bddk_housing_credit_stock_million_tl,BIGINT
92,3,finturk_domestic_provinces_million_try,DOUBLE
93,4,finturk_abroad_million_try,DOUBLE
94,5,finturk_all_geographies_million_try,DOUBLE
...,...,...,...
169,80,TP_MK_F_BILESIK,DOUBLE
170,81,TP_KKM_K1,DOUBLE
171,82,TP_KKM_K2,DOUBLE
172,83,TP_KKM_K4,DOUBLE



### bddk.finturk_branch_zero_fallback_audit


...

,ordinal_position,column_name,data_type
1977,1,province_name,VARCHAR
1978,2,province_key,VARCHAR
1979,3,month,VARCHAR
1980,4,metric_code,VARCHAR
1981,5,evds_value,DOUBLE
1982,6,direct_value,DOUBLE
1983,7,tuik_usable_value,DOUBLE
1984,8,value_origin,VARCHAR
1985,9,source_csv_sha256,VARCHAR
1986,10,difference_direct,DOUBLE



### tuik.province_housing_sales_identity_zero_fallbacks


,ordinal_position,column_name,data_type
1988,1,province_name,VARCHAR
1989,2,province_key,VARCHAR
1990,3,month,VARCHAR
1991,4,metric_code,VARCHAR
1992,5,direct_value,DOUBLE
1993,6,value,DOUBLE
1994,7,value_origin,VARCHAR
1995,8,derivation,VARCHAR
1996,9,source_row_present,BOOLEAN
1997,10,is_identity_derived,BOOLEAN



### tuik.province_housing_sales_monthly


,ordinal_position,column_name,data_type
2004,1,province_name,VARCHAR
2005,2,province_key,VARCHAR
2006,3,month,VARCHAR
2007,4,metric_code,VARCHAR
2008,5,direct_value,DOUBLE
2009,6,value,DOUBLE
2010,7,value_origin,VARCHAR
2011,8,derivation,VARCHAR
2012,9,source_row_present,BOOLEAN
2013,10,is_identity_derived,BOOLEAN


In [5]:
def null_profile(qualified_name):
    cols = columns.loc[
        (columns.table_schema + '.' + columns.table_name).eq(qualified_name), 'column_name'
    ].tolist()
    total = con.sql(f'SELECT count(*) FROM {qualified_name}').fetchone()[0]
    rows = []
    for col in cols:
        nulls = con.sql(f'SELECT count(*) FROM {qualified_name} WHERE "{col}" IS NULL').fetchone()[0]
        rows.append({'table': qualified_name, 'column': col, 'null_count': nulls, 'null_rate': nulls / total if total else 0})
    return pd.DataFrame(rows)
for target in ['analysis.housing_credit_monthly', 'analysis.housing_credit_quarterly', 'regional.housing_quarterly']:
    if target in tables.qualified_name.values:
        print(target)
        display(null_profile(target).sort_values('null_rate', ascending=False))

analysis.housing_credit_monthly


,table,column,null_count,null_rate
89,analysis.housing_credit_monthly,event_verification_levels,61,0.924242
88,analysis.housing_credit_monthly,event_titles,61,0.924242
87,analysis.housing_credit_monthly,event_ids,61,0.924242
45,analysis.housing_credit_monthly,TP_HANEBEK_HAN18A,60,0.909091
53,analysis.housing_credit_monthly,TP_PKAUO_S04_C_U,53,0.803030
...,...,...,...,...
79,analysis.housing_credit_monthly,housing_credit_real_million_tl_jan2021_prices,0,0.000000
85,analysis.housing_credit_monthly,rate_down_real_stock_not_up_quality_screened,0,0.000000
84,analysis.housing_credit_monthly,rate_down_real_stock_not_up,0,0.000000
83,analysis.housing_credit_monthly,mortgaged_sales_share_pct,0,0.000000


analysis.housing_credit_quarterly


,table,column,null_count,null_rate
50,analysis.housing_credit_quarterly,TP_HANEBEK_HAN18A,20,0.909091
58,analysis.housing_credit_quarterly,TP_PKAUO_S04_C_U,17,0.772727
80,analysis.housing_credit_quarterly,TP_KKM_K1,3,0.136364
81,analysis.housing_credit_quarterly,TP_KKM_K2,3,0.136364
82,analysis.housing_credit_quarterly,TP_KKM_K4,3,0.136364
...,...,...,...,...
72,analysis.housing_credit_quarterly,TP_GSYIH60_HY_B1GQ,0,0.000000
79,analysis.housing_credit_quarterly,TP_MK_F_BILESIK,0,0.000000
77,analysis.housing_credit_quarterly,TP_ALTINPIYASA_KAP02,0,0.000000
76,analysis.housing_credit_quarterly,DERIVED_BIST_GOLD_TL_GR,0,0.000000


regional.housing_quarterly


,table,column,null_count,null_rate
39,regional.housing_quarterly,housing_unit_price_yoy_pct,466,0.261504
41,regional.housing_quarterly,regional_kfe_yoy_pct,324,0.181818
40,regional.housing_quarterly,housing_unit_price_for_analysis_yoy_pct,324,0.181818
42,regional.housing_quarterly,regional_ykke_yoy_pct,324,0.181818
43,regional.housing_quarterly,housing_credit_yoy_pct,324,0.181818
16,regional.housing_quarterly,housing_unit_price_try_per_m2,162,0.090909
0,regional.housing_quarterly,province_key,0,0.000000
1,regional.housing_quarterly,province_name,0,0.000000
8,regional.housing_quarterly,housing_sales_total_count,0,0.000000
9,regional.housing_quarterly,sales_evds_source_complete,0,0.000000


Bu tabloda yüksek NULL oranı, otomatik olarak veri hatası demek değildir. Örneğin TBB’nin yayımlanmayan 2026-06 dönemi veya bazı illerde kaynakta bulunmayan fiyat serileri NULL görünür. Önce `source_gap`, `analysis_ready` ve kaynak kapsamı kolonlarını kontrol etmeliyiz. `analysis` tablosundaki temel aylık değişkenler NULL değilse ulusal analiz yapılabilir; bölgesel paneldeki NULL’lar ise il/dönem bazında filtrelenmelidir.

## 3. Mantıksal foreign key ilişkileri
Bu ilişkiler fiziksel constraint değil, veri modelinin beklenen anahtarlarıdır. Orphan sayısı sıfır olmalıdır.

In [6]:
def has_table(name):
    return name in set(tables.qualified_name)

relationships = [
    ('bddk.monthly_measurements', 'bddk.monthly_metric_dictionary', ['table_no', 'metric_code'], ['table_no', 'metric_code'], 'BDDK aylık ölçüm -> metrik sözlüğü'),
    ('bddk.weekly_measurements', 'bddk.weekly_metric_dictionary', ['table_id', 'metric_code'], ['table_id', 'metric_code'], 'BDDK haftalık ölçüm -> metrik sözlüğü'),
    ('bddk.weekly_measurements', 'bddk.weekly_source_tables', ['observation_date', 'table_id', 'group_code'], ['observation_date', 'table_id', 'group_code'], 'BDDK haftalık ölçüm -> kaynak tablo'),
    ('bddk.finturk_measurements', 'bddk.finturk_metric_dictionary', ['measure_code'], ['measure_code'], 'FinTürk ölçüm -> metrik sözlüğü'),
    ('bddk.finturk_measurements', 'bddk.finturk_source_tables', ['quarter', 'table_no', 'source_file', 'source_sha256'], ['quarter', 'table_no', 'source_file', 'source_sha256'], 'FinTürk ölçüm -> kaynak tablo'),
    ('regional.housing_quarterly', 'regional.province_dimension', ['province_key'], ['province_key'], 'İl paneli -> il boyutu'),
    ('tuik.province_housing_sales_evds_reconciliation', 'tuik.province_housing_sales_monthly', ['province_key', 'month', 'metric_code'], ['province_key', 'month', 'metric_code'], 'TÜİK uzlaştırma -> TÜİK aylık veri'),
]


def orphan_count(child, parent, child_keys, parent_keys):
    if not has_table(child) or not has_table(parent):
        return None
    condition = ' AND '.join(f'c."{a}" = p."{b}"' for a, b in zip(child_keys, parent_keys))
    return con.sql(f'SELECT count(*) FROM {child} c ANTI JOIN {parent} p ON {condition}').fetchone()[0]

relationship_rows = []
for child, parent, child_keys, parent_keys, description in relationships:
    count = orphan_count(child, parent, child_keys, parent_keys)
    relationship_rows.append({'relationship': description, 'child': child, 'parent': parent, 'orphan_rows': count, 'status': 'SKIP' if count is None else ('PASS' if count == 0 else 'CHECK')})

for observation_table in sorted(name for name in tables.qualified_name if name.startswith('evds.') and name.endswith('_observations')):
    count = orphan_count(observation_table, 'evds.series_catalog', ['series_code'], ['series_code'])
    relationship_rows.append({'relationship': 'EVDS observation -> seri kataloğu', 'child': observation_table, 'parent': 'evds.series_catalog', 'orphan_rows': count, 'status': 'SKIP' if count is None else ('PASS' if count == 0 else 'CHECK')})

relationship_df = pd.DataFrame(relationship_rows)
display(relationship_df)

,relationship,child,parent,orphan_rows,status
0,BDDK aylık ölçüm -> metrik sözlüğü,bddk.monthly_measurements,bddk.monthly_metric_dictionary,0,PASS
1,BDDK haftalık ölçüm -> metrik sözlüğü,bddk.weekly_measurements,bddk.weekly_metric_dictionary,0,PASS
2,BDDK haftalık ölçüm -> kaynak tablo,bddk.weekly_measurements,bddk.weekly_source_tables,0,PASS
3,FinTürk ölçüm -> metrik sözlüğü,bddk.finturk_measurements,bddk.finturk_metric_dictionary,0,PASS
4,FinTürk ölçüm -> kaynak tablo,bddk.finturk_measurements,bddk.finturk_source_tables,0,PASS
5,İl paneli -> il boyutu,regional.housing_quarterly,regional.province_dimension,0,PASS
6,TÜİK uzlaştırma -> TÜİK aylık veri,tuik.province_housing_sales_evds_reconciliation,tuik.province_housing_sales_monthly,0,PASS
7,EVDS observation -> seri kataloğu,evds.household_finance_observations,evds.series_catalog,0,PASS
8,EVDS observation -> seri kataloğu,evds.housing_controls_observations,evds.series_catalog,0,PASS
9,EVDS observation -> seri kataloğu,evds.housing_observations,evds.series_catalog,0,PASS


## Çıktının yorumu
`status=PASS` ve `orphan_rows=0`, ölçüm tablolarındaki anahtarların ilgili sözlük/kaynak tablolarında karşılığı olduğunu gösterir. `SKIP`, tablonun mevcut olmadığını; `CHECK`, eşleşmeyen kayıt bulunduğunu anlatır. PASS sonucu verinin ekonomik olarak doğru olduğunu değil, veri modelindeki teknik bağlantının kopuk olmadığını kanıtlar.

In [7]:
display(con.sql("""
    SELECT source_system,
           count(*) AS total_metrics,
           count(*) FILTER (WHERE observation_available) AS available_metrics,
           round(100.0 * count(*) FILTER (WHERE observation_available) / NULLIF(count(*), 0), 2) AS coverage_pct
    FROM catalog.metrics
    GROUP BY source_system
    ORDER BY source_system
""").df())
display(con.sql("""
    SELECT count(*) AS total_metrics,
           count(*) FILTER (WHERE observation_available) AS queryable_metrics,
           round(100.0 * count(*) FILTER (WHERE observation_available) / NULLIF(count(*), 0), 2) AS queryable_pct
    FROM catalog.metrics
""").df())

,source_system,total_metrics,available_metrics,coverage_pct
0,BDDK_FINTURK,76,76,100.00
1,BDDK_MONTHLY,2108,2108,100.00
2,BDDK_WEEKLY,517,515,99.61
3,REGIONAL_HOUSING_ANALYSIS,31,30,96.77
4,TBB_REPORTS,60,60,100.00
5,TCMB_EVDS,52696,506,0.96
6,TCMB_EVDS_DERIVED,1,1,100.00
7,TUIK_DATA_PORTAL,5,5,100.00


,total_metrics,queryable_metrics,queryable_pct
0,55494,3301,5.95


## Çıktının yorumu
`coverage_pct`, katalogdaki metriklerin ne kadarının yerel gözleme sahip olduğunu gösterir. EVDS için oran düşük görünüyorsa bu beklenen bir durumdur: 52.696 serinin metadata’sı katalogda bulunur, fakat yalnızca analiz için seçilen seri grubunun gözlemleri indirilmiştir. Bu oranı veri kalitesi değil, yerel gözlem kapsamı olarak yorumlamak gerekir.

## 4. Basit analiz 
Aşağıdaki sorgular lakehouse’taki veri ailelerinin ne anlatabildiğini gösterir.

In [8]:
display(con.sql("""
    SELECT month,
           round(bddk_housing_credit_stock_million_tl, 2) AS housing_stock_million_tl,
           round(housing_credit_real_mom_pct, 2) AS real_stock_mom_pct,
           round(TP_KTF12, 2) AS housing_rate_pct,
           round(mortgaged_sales_share_pct, 2) AS mortgaged_sales_share_pct
    FROM analysis.housing_credit_monthly
    ORDER BY month DESC
    LIMIT 12
""").df())

,month,housing_stock_million_tl,real_stock_mom_pct,housing_rate_pct,mortgaged_sales_share_pct
0,2026-06,801438,1.37,40.34,20.00
1,2026-05,782918,0.31,38.43,21.17
2,2026-04,767314,-0.74,36.62,20.32
3,2026-03,741972,1.69,34.66,22.91
4,2026-02,715799,0.56,35.16,20.10
5,2026-01,691343,-2.88,36.24,18.18
6,2025-12,678970,2.16,37.29,11.54
7,2025-11,658770,1.71,37.76,15.32
8,2025-10,642144,0.39,37.94,14.44
9,2025-09,623765,-0.73,38.80,14.19


## Çıktının yorumu
Bu çıktı son 12 ayda konut kredisi stokunu, reel aylık değişimi, kredi faizini ve ipotekli satış payını yan yana gösterir. Reel stok artmıyor ve faiz değişimi negatifse bu yalnızca dikkat çekici bir dönemdir; kredi talebi, arz koşulları, enflasyon ve banka kapsamı kontrol edilmeden faiz düşüşünün etkisi olarak yorumlanamaz.

In [9]:
display(con.sql("""
    SELECT quarter,
           round(bddk_housing_credit_stock_million_tl, 2) AS bddk_stock,
           round(finturk_domestic_provinces_million_try, 2) AS finturk_domestic_stock,
           round(balance_amount_million_try, 2) AS tbb_balance,
           round(disbursement_amount_million_try, 2) AS tbb_disbursement
    FROM analysis.housing_credit_quarterly
    ORDER BY quarter DESC
    LIMIT 8
""").df())

,quarter,bddk_stock,finturk_domestic_stock,tbb_balance,tbb_disbursement
0,2026-06,801438,801440.79,NaN,NaN
1,2026-03,741972,741975.63,675178.32,99520.74
2,2025-12,678970,678974.58,614238.89,86982.71
3,2025-09,623765,623770.00,562890.17,68307.00
4,2025-06,583265,583270.06,525421.65,58246.68
5,2025-03,548385,548390.60,495614.44,55340.64
6,2024-12,512259,512264.82,464901.93,67391.65
7,2024-09,465313,465320.39,420965.71,36915.69


## Çıktının yorumu
BDDK, FinTürk ve TBB satırlarını doğrudan eşit kabul etmeyin. BDDK sektör stokunu, FinTürk il toplamını, TBB ise raporlayan bankaların bakiyesini ve gerçek kullandırım akımını gösterir. TBB’nin 2026-06 değeri kaynakta yoksa NULL kalması beklenir; bu bir sıfır veya tahmin değildir.

In [10]:
display(con.sql("""
    SELECT province_name, quarter,
           housing_sales_total_count,
           housing_sales_mortgaged_count,
           round(mortgaged_sales_share_pct, 2) AS mortgaged_share_pct,
           analysis_ready
    FROM regional.housing_quarterly
    WHERE quarter = '2026Q2'
    ORDER BY housing_sales_total_count DESC
    LIMIT 15
""").df())

,province_name,quarter,housing_sales_total_count,housing_sales_mortgaged_count,mortgaged_share_pct,analysis_ready
0,İSTANBUL,2026Q2,66133.0,15328.0,23.18,True
1,ANKARA,2026Q2,31169.0,7948.0,25.50,True
2,İZMİR,2026Q2,20704.0,4558.0,22.02,True
3,ANTALYA,2026Q2,17569.0,2655.0,15.11,True
4,MERSİN,2026Q2,11933.0,2055.0,17.22,True
5,BURSA,2026Q2,11842.0,2817.0,23.79,True
6,KOCAELİ,2026Q2,9872.0,2662.0,26.97,True
7,GAZİANTEP,2026Q2,9715.0,1896.0,19.52,True
8,TEKİRDAĞ,2026Q2,9010.0,2270.0,25.19,True
9,KONYA,2026Q2,8040.0,1401.0,17.43,True


## Çıktının yorumu
Bu sıralama 2026Q2’de toplam konut satışı yüksek olan illeri gösterir. `mortgaged_share_pct` ipotekli satışların toplam satış içindeki payıdır; `analysis_ready=True` satır seçilmiş kaynakların analiz için kullanılabilir olduğunu belirtir. Büyük satış hacmi tek başına yüksek kredi yoğunluğu veya yüksek fiyat anlamına gelmez.

In [11]:
display(con.sql("""
    SELECT reconciliation_status, count(*) AS observation_count
    FROM tuik.province_housing_sales_evds_reconciliation
    GROUP BY reconciliation_status
    ORDER BY reconciliation_status
""").df())
display(con.sql("""
    SELECT m.group_code, s.group_name, count(*) AS measurement_rows,
           round(avg(m.value), 2) AS average_value
    FROM bddk.weekly_measurements m
    JOIN bddk.weekly_source_tables s
      ON s.observation_date = m.observation_date
     AND s.table_id = m.table_id
     AND s.group_code = m.group_code
    GROUP BY m.group_code, s.group_name
    ORDER BY m.group_code
""").df())

,reconciliation_status,observation_count
0,evds_null_tuik_identity_zero,10
1,exact_match,25262


,group_code,group_name,measurement_rows,average_value
0,10001,Sektor,147154,762424.13
1,10002,Mevduat,146470,676206.04
2,10003,Kalkinma ve Yatirim,146470,31346.98
3,10004,Katilim,146470,53085.06
4,10005,Kamu,146470,297976.85
5,10006,Yabanci,146470,220856.95
6,10007,Yerli Ozel,146470,241803.77


## Çıktının yorumu
TÜİK-EVDS `exact_match` sonuçlarının baskın olması iki resmî kaynağın ortak gözlemlerde aynı değerleri verdiğini gösterir. `value_mismatch` varsa önce tarih, il, metrik kodu ve kaynak sürümü incelenmelidir. BDDK grup ortalamaları ise farklı tablo ve ölçüleri birlikte içerdiğinden yalnızca kapsamı görmek için kullanılmalı; ekonomik büyüklük karşılaştırması aynı metrik seçilerek yapılmalıdır.

## 5. Bu verilerle ne yapılabilir?

- Ulusal aylık konut kredisi stoku, faiz, enflasyon ve satış ilişkileri incelenebilir.
- Banka grupları arasında haftalık/aylık büyüklük ve kompozisyon karşılaştırılabilir.
- İl bazında konut satışları, fiyatlar, kredi ve mevduat göstergeleri çeyreklik olarak karşılaştırılabilir.
- EVDS ve TÜİK aynı göstergeler için çapraz doğrulama kaynağı olarak kullanılabilir.
- TBB kullandırım akımı ile BDDK/FinTürk stokları aynı şey değildir; birlikte kullanılırken kapsam ve anlam farkları korunmalıdır.

## Dikkat edilmesi gerekenler

- TBB `2026-06` raporu kaynakta yayımlanmadığı için boş kalır.
- Bazı illerde konut birim fiyatı kaynakta hiç yoktur; bu değerler tahmin edilmemiştir.
- EVDS kataloğunda 52.696 seri vardır, fakat yerel gözlemler seçilmiş serilerle sınırlıdır.
- Birlikte hareket korelasyon veya nedensellik kanıtı değildir.
- `analysis` tabloları hazır analiz görünümleridir; ham kaynakların tamamı değildir.

## sonraki analizler

1. İl bazında konut satış payı ve kredi yoğunluğu kümeleri.
2. Banka grubu bazında kredi kompozisyonu ve büyüme karşılaştırması.
3. Faiz değişimlerinden sonra kredi stokunun gecikmeli tepkisi.
4. Kaynak kapsamı uyarılarını dikkate alan panel regresyonu.
5. Anomali ve değişim noktası tespiti.

## il kümeleri
İl bazında satış payı ve kredi yoğunluğunu son çeyrek için karşılaştırıyoruz. `qcut` ile eşit gözlem sayısına yakın düşük/orta/yüksek kümeleri oluşturulur; bu bir segmentasyon özetidir, makine öğrenmesi sonucu değildir.

In [12]:
regional = con.sql("""
    SELECT province_name, quarter,
           housing_sales_total_count,
           mortgaged_sales_share_pct,
           housing_credit_per_capita_try,
           regional_kfe_index,
           regional_ykke_index
    FROM regional.housing_quarterly
    WHERE quarter = (SELECT max(quarter) FROM regional.housing_quarterly)
      AND analysis_ready
""").df()
for column in ['mortgaged_sales_share_pct', 'housing_credit_per_capita_try']:
    regional[f'{column}_cluster'] = pd.qcut(
        regional[column], q=3, labels=['düşük', 'orta', 'yüksek'], duplicates='drop'
    )
regional['sales_rank'] = regional['housing_sales_total_count'].rank(ascending=False, method='min').astype('Int64')
print(f"Kümeleme dönemi: {regional['quarter'].iloc[0]} | İl sayısı: {len(regional)}")
display(regional.sort_values(['housing_credit_per_capita_try', 'housing_sales_total_count'], ascending=False).head(20))
display(
    regional.groupby(['mortgaged_sales_share_pct_cluster', 'housing_credit_per_capita_try_cluster'], observed=True)
    .agg(province_count=('province_name', 'count'), average_sales=('housing_sales_total_count', 'mean'), average_credit_per_capita=('housing_credit_per_capita_try', 'mean'))
    .reset_index()
)

Kümeleme dönemi: 2026Q2 | İl sayısı: 81


,province_name,quarter,housing_sales_total_count,mortgaged_sales_share_pct,housing_credit_per_capita_try,regional_kfe_index,regional_ykke_index,mortgaged_sales_share_pct_cluster,housing_credit_per_capita_try_cluster,sales_rank
6,ANKARA,2026Q2,31169.0,25.499695,18430.067383,255.69,352.04,yüksek,yüksek,2
31,ESKİŞEHİR,2026Q2,5690.0,22.179262,15945.319583,227.71,309.47,yüksek,yüksek,18
39,İSTANBUL,2026Q2,66133.0,23.177536,14496.807320,216.62,304.25,yüksek,yüksek,1
72,TEKİRDAĞ,2026Q2,9010.0,25.194229,13858.408463,216.84,312.50,yüksek,yüksek,9
40,İZMİR,2026Q2,20704.0,22.015070,13314.289032,225.16,323.51,yüksek,yüksek,3
51,KOCAELİ,2026Q2,9872.0,26.965154,11736.393832,235.64,332.72,yüksek,yüksek,7
24,DENİZLİ,2026Q2,4150.0,24.433735,11435.100715,222.22,315.63,yüksek,yüksek,22
21,ÇANAKKALE,2026Q2,2878.0,16.817234,10934.054724,238.31,331.06,düşük,yüksek,26
7,ANTALYA,2026Q2,17569.0,15.111845,9967.780264,194.13,255.05,düşük,yüksek,4
48,KIRIKKALE,2026Q2,1299.0,25.404157,9804.009462,254.08,359.33,yüksek,yüksek,52


,mortgaged_sales_share_pct_cluster,housing_credit_per_capita_try_cluster,province_count,average_sales,average_credit_per_capita
0,düşük,düşük,16,1677.187500,2860.437388
1,düşük,orta,5,830.200000,6050.324061
2,düşük,yüksek,6,6047.333333,9206.939781
3,orta,düşük,9,2631.444444,3753.953369
4,orta,orta,12,2937.000000,5967.851063
5,orta,yüksek,6,4963.833333,8678.818541
6,yüksek,düşük,2,1974.000000,3641.826609
7,yüksek,orta,10,2063.800000,6291.181113
8,yüksek,yüksek,15,11303.600000,11464.499036


## Çıktının yorumu
Bu çıktı illeri son çeyrekteki iki boyuta göre düşük/orta/yüksek segmentlere ayırır. Örneğin yüksek kredi yoğunluğu ve yüksek ipotekli satış payı aynı grupta olan iller, daha ileri inceleme için adaydır. `qcut` göreli bir sıralama yapar; kümeler arasında nedensel fark veya ekonomik eşik olduğu anlamına gelmez.

## banka grubu karşılaştırması
Haftalık BDDK ölçümlerini banka grubu ve para boyutunda özetliyoruz. Ölçüler farklı tabloları içerdiği için doğrudan tüm metriklerin ortalaması ekonomik gösterge olarak yorumlanmamalıdır.

In [13]:
bank_groups = con.sql("""
    SELECT m.group_code, s.group_name, m.table_id,
           m.currency_dimension,
           m.observation_date,
           sum(m.value) AS total_value,
           count(*) AS measurement_count
    FROM bddk.weekly_measurements m
    LEFT JOIN bddk.weekly_source_tables s
      ON s.observation_date = m.observation_date
     AND s.table_id = m.table_id
     AND s.group_code = m.group_code
    WHERE m.value IS NOT NULL
    GROUP BY m.group_code, s.group_name, m.table_id, m.currency_dimension, m.observation_date
""").df()
bank_summary = (
    bank_groups.groupby(['group_code', 'group_name'], as_index=False)
    .agg(average_group_value=('total_value', 'mean'), total_measurements=('measurement_count', 'sum'), observed_weeks=('observation_date', 'nunique'))
    .sort_values('average_group_value', ascending=False)
)
display(bank_summary)
print('Karşılaştırılabilir bir metrik için aynı table_id, metric_code ve currency_dimension seçmek gerekir.')

,group_code,group_name,average_group_value,total_measurements,observed_weeks
0,10001,Sektor,1.563663e+07,146640,286
1,10002,Mevduat,1.382525e+07,146184,286
4,10005,Kamu,6.092230e+06,146184,286
6,10007,Yerli Ozel,4.943754e+06,146184,286
5,10006,Yabanci,4.515490e+06,146184,286
3,10004,Katilim,1.085341e+06,146184,286
2,10003,Kalkinma ve Yatirim,6.408989e+05,146184,286


Karşılaştırılabilir bir metrik için aynı table_id, metric_code ve currency_dimension seçmek gerekir.


## Çıktının yorumu
Bu çalıştırmada 7 banka grubu ve her grup için 286 haftalık gözlem görüldü; yani haftalık grup kapsamı eksiksiz görünüyor. Ham ortalama değer sıralaması Sektör, Mevduat, Kamu, Yerli Özel, Yabancı, Katılım ve Kalkınma-Yatırım şeklinde çıktı. Sektör toplamının tek bir banka grubu olmadığını, diğer grupların toplam kapsamıyla karşılaştırıldığını unutmayın. Bu sıralama farklı `table_id` ve `metric_code` değerleri birlikte toplandığı için ekonomik büyüklük sıralaması değildir; gerçek karşılaştırmada tek bir aynı metrik seçilmelidir.

## faiz sonrası gecikmeli tepki
Faiz değişimi ile reel konut kredisi stok değişiminin 1, 3 ve 6 aylık gecikmeli birlikte hareketi hesaplanır. Bu korelasyon nedensellik değildir; yalnızca ileri analiz için aday gecikmeleri gösterir.

In [14]:
monthly = con.sql("""
    SELECT month, housing_rate_change_pp, housing_credit_real_mom_pct
    FROM analysis.housing_credit_monthly
    ORDER BY month
""").df()
monthly['month'] = pd.to_datetime(monthly['month'] + '-01')
lag_rows = []
for lag in [1, 3, 6]:
    frame = monthly.copy()
    frame['future_real_stock_change'] = frame['housing_credit_real_mom_pct'].shift(-lag)
    frame = frame.dropna(subset=['housing_rate_change_pp', 'future_real_stock_change'])
    lag_rows.append({'lag_months': lag, 'observations': len(frame), 'correlation_rate_change_vs_future_real_stock': frame['housing_rate_change_pp'].corr(frame['future_real_stock_change'])})
lag_results = pd.DataFrame(lag_rows)
display(lag_results)
print('Yorum: Bu sonuçlar gecikmeli birlikte hareketi tarar; politika etkisi veya nedensellik iddiası değildir.')

,lag_months,observations,correlation_rate_change_vs_future_real_stock
0,1,64,-0.373916
1,3,62,-0.274657
2,6,59,-0.166307


Yorum: Bu sonuçlar gecikmeli birlikte hareketi tarar; politika etkisi veya nedensellik iddiası değildir.


## Çıktının yorumu
Korelasyon katsayısı pozitifse faiz değişimi ile gelecekteki reel stok değişimi aynı yönde, negatifse ters yönde hareket etme eğilimi vardır. Ancak 66 aylık kısa seri, enflasyon, arz, düzenleme ve dönemsel kapsam farkları nedeniyle bu tablo yalnızca hangi gecikmelerin araştırılmaya değer olduğunu gösterir.

## kaynak kapsamını dikkate alan panel regresyonu
İl-çeyrek panelinde il ortalamalarını çıkararak basit bir within dönüşümü uygulanır. Bu, il sabit etkilerini kabaca kontrol eden keşif analizidir; eksik fiyat gözlemleri ve kaynak kapsam farkları nedeniyle nedensel model olarak yorumlanmamalıdır.

In [15]:
panel = con.sql("""
    SELECT province_key, quarter,
           housing_sales_total_count AS sales_total,
           housing_credit_per_capita_try AS credit_per_capita,
           mortgaged_sales_share_pct AS mortgaged_share,
           regional_kfe_index AS kfe,
           regional_ykke_index AS ykke
    FROM regional.housing_quarterly
    WHERE analysis_ready
""").df().dropna()
features = ['credit_per_capita', 'mortgaged_share', 'kfe', 'ykke']
for column in ['sales_total', *features]:
    panel[f'{column}_demeaned'] = panel[column] - panel.groupby('province_key')[column].transform('mean')
model_data = panel[['sales_total_demeaned', *[f'{column}_demeaned' for column in features]]]
X = model_data[[f'{column}_demeaned' for column in features]].to_numpy()
y = model_data['sales_total_demeaned'].to_numpy()
X_design = __import__('numpy').column_stack([__import__('numpy').ones(len(X)), X])
coefficients, _, _, _ = __import__('numpy').linalg.lstsq(X_design, y, rcond=None)
predicted = X_design @ coefficients
residual = y - predicted
regression_summary = pd.DataFrame({'term': ['intercept', *features], 'coefficient': coefficients})
print(f'Gözlem sayısı: {len(model_data)} | İl sayısı: {panel.province_key.nunique()}')
display(regression_summary)
print(f'Within R-kare: {1 - (residual @ residual) / ((y - y.mean()) @ (y - y.mean())):.4f}')
print('Katsayılar keşif amaçlıdır; seriler farklı kapsam ve ölçüm anlamlarına sahip olabilir.')

Gözlem sayısı: 1782 | İl sayısı: 81


,term,coefficient
0,intercept,-1.939049e-13
1,credit_per_capita,2.952025e-01
2,mortgaged_share,-1.397267e+01
3,kfe,-3.425598e+01
4,ykke,2.035383e+01


Within R-kare: 0.0376
Katsayılar keşif amaçlıdır; seriler farklı kapsam ve ölçüm anlamlarına sahip olabilir.


## Çıktının yorumu
Within katsayıları, her ilin kendi ortalamasından sapmalarla satış hacmi arasındaki ilişkiyi özetler. Pozitif katsayı birlikte artış, negatif katsayı ters yönlü ilişki demektir. `R-kare` modelin açıklama gücünü gösterir; istatistiksel anlamlılık, standart hata ve zaman sabit etkileri hesaplanmadığı için bu sonuç karar veya nedensellik kanıtı değildir.

## anomali ve değişim noktası taraması
Aylık reel kredi stoku değişiminde rolling median ve MAD tabanlı robust z-score kullanılır. Büyük mutlak skorlar incelenmesi gereken dönemleri işaretler; tek başına ekonomik açıklama değildir.

In [16]:
series = con.sql("""
    SELECT month, housing_credit_real_mom_pct, housing_rate_change_pp
    FROM analysis.housing_credit_monthly
    ORDER BY month
""").df()
series['rolling_median'] = series['housing_credit_real_mom_pct'].rolling(6, min_periods=3).median()
series['rolling_mad'] = series['housing_credit_real_mom_pct'].rolling(6, min_periods=3).apply(lambda values: (values - values.median()).abs().median(), raw=False)
series['robust_z'] = (series['housing_credit_real_mom_pct'] - series['rolling_median']) / (1.4826 * series['rolling_mad'].replace(0, pd.NA))
series['change_point_flag'] = series['robust_z'].abs().ge(3)
print(f"İşaretlenen dönem sayısı: {int(series['change_point_flag'].sum())}")
display(series.loc[series['change_point_flag'], ['month', 'housing_credit_real_mom_pct', 'housing_rate_change_pp', 'robust_z']])
display(series.sort_values('robust_z', key=lambda values: values.abs(), ascending=False).head(10))

İşaretlenen dönem sayısı: 4


,month,housing_credit_real_mom_pct,housing_rate_change_pp,robust_z
11,2021-12,-9.762532,0.2105,-9.038560
12,2022-01,-9.461070,0.9370,-5.093523
38,2024-03,-1.711916,1.3370,4.272488
58,2025-11,1.709330,-0.1775,4.203736


,month,housing_credit_real_mom_pct,housing_rate_change_pp,rolling_median,rolling_mad,robust_z,change_point_flag
11,2021-12,-9.762532,0.2105,-1.365367,0.626627,-9.038560,True
12,2022-01,-9.461070,0.9370,-1.365367,1.072043,-5.093523,True
38,2024-03,-1.711916,1.3370,-3.846496,0.336983,4.272488,True
58,2025-11,1.709330,-0.1775,0.155858,0.249255,4.203736,True
3,2021-04,-1.779580,0.3990,-1.086581,0.174493,-2.678737,False
26,2023-03,4.110455,-3.4065,-2.043693,1.590250,2.610229,False
45,2024-10,0.367362,-1.6325,-1.348000,0.642238,1.801507,False
60,2026-01,-2.878626,-1.0520,0.356826,1.222049,-1.785757,False
36,2024-01,-6.610255,-0.5105,-4.390518,0.852143,-1.756972,False
7,2021-08,-0.334865,-0.0715,-1.212373,0.338017,1.751012,False


## Çıktının yorumu
`change_point_flag=True` olan aylar, son 6 aylık hareketli dağılıma göre reel kredi değişiminin olağandışı olduğu dönemlerdir. Bunlar inceleme adaylarıdır; sebebi faiz, düzenleme, enflasyon, kaynak revizyonu veya veri kapsamı olabilir. İlk ve son gözlemlerde rolling pencere kısa olduğu için bayraklar daha temkinli yorumlanmalıdır.

## 11. AI olmadan doğrudan SQL sorgulama
Bu bölüm, agent olmadan bir veri uzmanının lakehouse’u doğrudan kullanabildiğini gösterir. Sorgu DuckDB içinde çalışır; sonuçlar doğrudan tablolar ve katalog üzerinden alınır.

In [17]:
direct_query = con.sql("""
    SELECT month,
           round(bddk_housing_credit_stock_million_tl, 2) AS housing_stock_million_tl,
           round(TP_KTF12, 2) AS housing_rate_pct,
           round(housing_credit_real_mom_pct, 2) AS real_stock_mom_pct
    FROM analysis.housing_credit_monthly
    WHERE month BETWEEN '2025-01' AND '2026-06'
    ORDER BY month
""").df()
print(f'Doğrudan sorgu sonucu: {len(direct_query)} satır')
display(direct_query)
print('Bu sorgu için AI gerekmez; kullanıcı veya veri uzmanı tablo/kolon adlarını bilerek SQL çalıştırabilir.')

Doğrudan sorgu sonucu: 18 satır


,month,housing_stock_million_tl,housing_rate_pct,real_stock_mom_pct
0,2025-01,523455,40.54,-2.71
1,2025-02,535029,39.97,-0.06
2,2025-03,548385,39.38,0.04
3,2025-04,561793,39.56,-0.54
4,2025-05,575453,42.14,0.89
5,2025-06,583265,43.47,-0.02
6,2025-07,594607,42.56,-0.11
7,2025-08,608694,40.36,0.33
8,2025-09,623765,38.80,-0.73
9,2025-10,642144,37.94,0.39


Bu sorgu için AI gerekmez; kullanıcı veya veri uzmanı tablo/kolon adlarını bilerek SQL çalıştırabilir.


## 12. Veri hizalama ve frekans uyumu
Aynı analiz dönemlerinin ulusal aylık, bölgesel çeyreklik ve EVDS panellerinde nasıl temsil edildiğini kontrol ediyoruz. Çeyreklik gözlemler ara aylara kopyalanmamalıdır.

In [18]:
alignment_checks = []
checks = [
    ('analysis monthly', "SELECT count(*) FROM analysis.housing_credit_monthly", 66),
    ('analysis quarterly', "SELECT count(*) FROM analysis.housing_credit_quarterly", 22),
    ('regional province-quarter', "SELECT count(*) FROM regional.housing_quarterly", 1782),
    ('regional distinct province-quarter', "SELECT count(DISTINCT province_key || ':' || quarter) FROM regional.housing_quarterly", 1782),
]
for name, query, expected in checks:
    actual = con.sql(query).fetchone()[0]
    alignment_checks.append({'check': name, 'actual': actual, 'expected': expected, 'status': 'PASS' if actual == expected else 'CHECK'})

quarterly_intermediate_months = con.sql("""
    SELECT count(*)
    FROM evds.housing_monthly_panel
    WHERE target_period LIKE '%-01'
       OR target_period LIKE '%-02'
       OR target_period LIKE '%-04'
       OR target_period LIKE '%-05'
       OR target_period LIKE '%-07'
       OR target_period LIKE '%-08'
       OR target_period LIKE '%-10'
       OR target_period LIKE '%-11'
""").fetchone()[0]
alignment_checks.append({'check': 'EVDS monthly panel structural scan', 'actual_intermediate_month_rows': quarterly_intermediate_months, 'status': 'INFO'})
display(pd.DataFrame(alignment_checks))
print('Açıklama: Çeyreklik serilerin ara aylarda değer taşıyıp taşımadığı, ilgili alignment audit tablolarında seri bazında incelenmelidir.')

,check,actual,expected,status,actual_intermediate_month_rows
0,analysis monthly,66.0,66.0,PASS,NaN
1,analysis quarterly,22.0,22.0,PASS,NaN
2,regional province-quarter,1782.0,1782.0,PASS,NaN
3,regional distinct province-quarter,1782.0,1782.0,PASS,NaN
4,EVDS monthly panel structural scan,NaN,NaN,INFO,52.0


Açıklama: Çeyreklik serilerin ara aylarda değer taşıyıp taşımadığı, ilgili alignment audit tablolarında seri bazında incelenmelidir.


## 13. BDDK kümülatif verilerinin ayrıştırılması
BDDK aylık semantik katmanında bazı kaynak değerler yıl içinde kümülatif ilerler. Builder bu değerleri aynı takvim yılı içinde farklayarak aylık akışa çevirir; Ocak ayı doğrudan kaynak değeri olarak korunur.

In [19]:
cumulative_result = con.sql("""
    SELECT
        count(*) AS cumulative_rows,
        count(*) FILTER (WHERE month LIKE '%-01') AS january_rows,
        max(abs(source_value - recomposed_value)) AS max_recomposition_difference
    FROM (
        SELECT table_no, metric_code, group_code, calendar_year, month,
               source_value,
               sum(analysis_value) OVER (
                   PARTITION BY table_no, metric_code, group_code, calendar_year
                   ORDER BY month
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
               ) AS recomposed_value
        FROM bddk.monthly_measurements
        WHERE transformation = 'difference_within_calendar_year'
    )
""").df()
display(cumulative_result)
validation_path = DB_PATH.parents[1] / 'bddk' / 'processed' / 'monthly_semantic' / 'validation.json'
if validation_path.exists():
    validation = pd.read_json(validation_path, typ='series')
    print('Builder validation özeti:')
    print(validation.get('recomposition', {}))
print('Beklenen sonuç: max_recomposition_difference sıfıra yakın olmalı; bu, aylık akışların kaynak kümülatif değerleri yeniden oluşturduğunu gösterir.')

,cumulative_rows,january_rows,max_recomposition_difference
0,104940,9540,0.0


Builder validation özeti:
{'cumulative_measurement_rows': 104940, 'recomposition_comparisons': 104940, 'maximum_recomposition_difference': 0.0, 'january_start_rows': 9540, 'annual_reset_pairs_checked': 7950, 'annual_reset_pairs_with_lower_january_absolute_value': 7069, 'annual_reset_share': 0.8891819999999999}
Beklenen sonuç: max_recomposition_difference sıfıra yakın olmalı; bu, aylık akışların kaynak kümülatif değerleri yeniden oluşturduğunu gösterir.


## 14. Veri kalitesi ve ilişkilendirilebilirlik özeti
Bu bölüm önceki NULL ve orphan kontrollerini tek bir kalite özeti hâline getirir: duplicate dönem, teknik orphan, kaynak uzlaştırma uyuşmazlığı ve beklenen kaynak boşlukları birlikte görülür.

In [20]:
quality_summary = []
quality_summary.append({'check': 'monthly duplicate periods', 'value': con.sql("SELECT count(*) - count(DISTINCT month) FROM analysis.housing_credit_monthly").fetchone()[0], 'expected': 0})
quality_summary.append({'check': 'quarterly duplicate periods', 'value': con.sql("SELECT count(*) - count(DISTINCT quarter) FROM analysis.housing_credit_quarterly").fetchone()[0], 'expected': 0})
if has_table('tuik.province_housing_sales_evds_reconciliation'):
    mismatch = con.sql("SELECT count(*) FROM tuik.province_housing_sales_evds_reconciliation WHERE reconciliation_status = 'value_mismatch'").fetchone()[0]
    quality_summary.append({'check': 'TÜİK-EVDS value mismatches', 'value': mismatch, 'expected': 0})
if has_table('tbb.source_gaps'):
    gaps = con.sql("SELECT count(*) FROM tbb.source_gaps").fetchone()[0]
    quality_summary.append({'check': 'declared TBB source gaps', 'value': gaps, 'expected': 'explicitly documented'})
quality_df = pd.DataFrame(quality_summary)
quality_df['status'] = quality_df.apply(lambda row: 'PASS' if row['expected'] == 0 and row['value'] == 0 else ('INFO' if row['expected'] != 0 else 'CHECK'), axis=1)
display(quality_df)
print('PASS teknik tutarlılığı; INFO/CHECK ise veri kaynağında açıklanmış kapsam boşluklarını veya incelenmesi gereken durumları gösterir.')

,check,value,expected,status
0,monthly duplicate periods,0,0,PASS
1,quarterly duplicate periods,0,0,PASS
2,TÜİK-EVDS value mismatches,0,0,PASS
3,declared TBB source gaps,1,explicitly documented,INFO


PASS teknik tutarlılığı; INFO/CHECK ise veri kaynağında açıklanmış kapsam boşluklarını veya incelenmesi gereken durumları gösterir.


## 15. Generik yapının doğrulanması
Lakehouse yalnızca konut kredisi kolonlarından oluşmuyor. `catalog.metrics`, `catalog.data_assets` ve `catalog.table_manifest` kaynak, frekans, birim, gözlem durumu ve dosya lineage’ını genel alanlarla taşıyor. Aşağıdaki sorgular finansal bir metriğin adını bilmeden çalışır.

In [21]:
generic_metrics = con.sql("""
    SELECT source_system,
           native_frequency,
           unit,
           count(*) AS metric_count,
           count(*) FILTER (WHERE observation_available) AS locally_queryable_count
    FROM catalog.metrics
    GROUP BY source_system, native_frequency, unit
    ORDER BY source_system, native_frequency, unit
""").df()
display(generic_metrics)

generic_assets = con.sql("""
    SELECT source_system,
           data_kind,
           native_frequency,
           count(*) AS asset_count,
           sum(row_count) AS total_rows
    FROM catalog.data_assets
    GROUP BY source_system, data_kind, native_frequency
    ORDER BY source_system, data_kind, native_frequency
""").df()
display(generic_assets)

manifest = con.sql("""
    SELECT schema_name, table_name, row_count, source_path
    FROM catalog.table_manifest
    ORDER BY schema_name, table_name
""").df()
display(manifest.head(30))
print('Generic sonuç: kaynak ve metrik seçimi katalogdan yapılabildiği için agent, finansal alan adlarına sabitlenmeden yeni veri ailelerini keşfedebilir.')

,source_system,native_frequency,unit,metric_count,locally_queryable_count
0,BDDK_FINTURK,quarterly,count,1,1
1,BDDK_FINTURK,quarterly,people_per_branch,1,1
2,BDDK_FINTURK,quarterly,percent,16,16
3,BDDK_FINTURK,quarterly,thousand_try,54,54
4,BDDK_FINTURK,quarterly,try_per_person,4,4
...,...,...,...,...,...
84,TCMB_EVDS,İŞ GÜNÜ,milyon ABD doları,38,0
85,TCMB_EVDS,İŞ GÜNÜ,milyon TL,2,0
86,TCMB_EVDS,İŞ GÜNÜ,milyon TL ve yüzde,12,1
87,TCMB_EVDS_DERIVED,derived,TL/gram,1,1


,source_system,data_kind,native_frequency,asset_count,total_rows
0,BDDK_FINTURK,identity_derived_zero_audit,quarterly,1,1328.0
1,BDDK_FINTURK,missingness_audit,quarterly,1,47.0
2,BDDK_FINTURK,observations_long,quarterly,1,936512.0
3,BDDK_FINTURK,source_table,quarterly,7,84484.0
4,BDDK_MONTHLY,semantic_observations_long,monthly,1,1334850.0
5,BDDK_MONTHLY,source_table,monthly,17,339650.0
6,BDDK_WEEKLY,missingness_audit,weekly,1,2.0
7,BDDK_WEEKLY,observations_long,weekly,1,1025974.0
8,BDDK_WEEKLY,raw_source_collection,weekly,1,18018.0
9,CROSS_SOURCE_QUALITY,cross_source_reconciliation,quarterly,1,22.0


,schema_name,table_name,row_count,source_path
0,analysis,housing_credit_monthly,66,"derived from validated BDDK, EVDS and event as..."
1,analysis,housing_credit_quarterly,22,"derived from validated BDDK, FinTurk, EVDS and..."
2,bddk,finturk_branch_zero_fallback_audit,1328,data_pipeline/bddk/processed/finturk_all_group...
3,bddk,finturk_measurements,936512,data_pipeline/bddk/processed/finturk_all_group...
4,bddk,finturk_metric_dictionary,76,data_pipeline/bddk/processed/finturk_all_group...
5,bddk,finturk_missingness_audit,47,data_pipeline/bddk/processed/finturk_all_group...
6,bddk,finturk_source_tables,154,data_pipeline/bddk/processed/finturk_all_group...
7,bddk,housing_credit_monthly,66,data_pipeline/bddk/processed/housing_credit_mo...
8,bddk,monthly_measurements,1334850,data_pipeline/bddk/processed/monthly_semantic/...
9,bddk,monthly_metric_dictionary,2108,data_pipeline/bddk/processed/monthly_semantic/...


Generic sonuç: kaynak ve metrik seçimi katalogdan yapılabildiği için agent, finansal alan adlarına sabitlenmeden yeni veri ailelerini keşfedebilir.


## Generik yapı çıktısının yorumu
`catalog.metrics` ve `catalog.data_assets` içindeki ortak alanlar, yeni bir veri ailesinin aynı keşif ve lineage sözleşmesine eklenebildiğini gösterir. Ancak bu henüz tamamen otomatik bir agent sistemi olduğu anlamına gelmez: yeni kaynağın downloader’ı, normalizasyon kuralı ve kalite testleri ayrıca yazılmalıdır. Generic olan kısım kataloglama, sorgulanabilirlik ve metadata modelidir; kaynak özelindeki semantik dönüşümler yine ayrı kurallara ihtiyaç duyar.